# EDA — XWines Slim Dataset
### Ratings + Content Features
---

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from collections import Counter
import ast
import warnings
warnings.filterwarnings('ignore')

sns.set(color_codes=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

import os
os.makedirs('figures', exist_ok=True)
print('Folder figures/ gotowy.')

## 1. Wczytanie danych

In [ ]:
df       = pd.read_csv('data/XWines_Slim_150K_ratings.csv')
df       = df[['UserID', 'WineID', 'Rating', 'Date']].copy()
wines_df = pd.read_csv('data/XWines_Slim_1K_wines.csv')

df['Date'] = pd.to_datetime(df['Date'])

print('=== Ratings ===')
print(f'Shape: {df.shape}')
print(df.dtypes)
print('\n=== Wines ===')
print(f'Shape: {wines_df.shape}')
print(wines_df.dtypes)

---
# CZĘŚĆ I — EDA RATINGS

## 2. Tabela zbiorcza — kluczowe metryki

In [ ]:
n_users   = df['UserID'].nunique()
n_wines   = df['WineID'].nunique()
n_ratings = len(df)
sparsity  = n_ratings / (n_users * n_wines)

summary = pd.DataFrame({
    'Parametr': [
        'Liczba ocen', 'Liczba użytkowników', 'Liczba win',
        'Sparsity macierzy [%]', 'Zakres dat (od)', 'Zakres dat (do)',
        'Średnia ocena', 'Mediana oceny',
        'Skośność (skewness)', 'Kurtoza'
    ],
    'Wartość': [
        f'{n_ratings:,}', f'{n_users:,}', f'{n_wines:,}',
        f'{sparsity*100:.4f}',
        df['Date'].min().strftime('%Y-%m-%d'),
        df['Date'].max().strftime('%Y-%m-%d'),
        f'{df["Rating"].mean():.4f}',
        f'{df["Rating"].median():.1f}',
        f'{df["Rating"].skew():.4f}',
        f'{df["Rating"].kurtosis():.4f}'
    ]
})
print(summary.to_string(index=False))

## 3. Rozkład ocen

In [ ]:
# Tabela procentowa
counts_abs  = df['Rating'].value_counts().sort_index()
counts_pct  = (counts_abs / counts_abs.sum() * 100).round(2)

rating_dist_df = pd.DataFrame({
    'Ocena':      counts_abs.index,
    'Liczba':     counts_abs.values,
    'Udział [%]': counts_pct.values
})
print(rating_dist_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
counts = df['Rating'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, width=0.4, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Ocena')
axes[0].set_ylabel('Liczba ocen')
axes[0].set_title('Rozkład ocen w zbiorze XWines Slim')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(0.5))

# Skumulowany CDF
sorted_ratings = np.sort(df['Rating'])
cdf = np.arange(1, len(sorted_ratings)+1) / len(sorted_ratings)
axes[1].plot(sorted_ratings, cdf, color='steelblue', linewidth=2)
axes[1].axvline(df['Rating'].mean(),   color='crimson', linestyle='--', label=f'Średnia = {df["Rating"].mean():.2f}')
axes[1].axvline(df['Rating'].median(), color='orange',  linestyle=':',  label=f'Mediana = {df["Rating"].median():.1f}')
axes[1].set_xlabel('Ocena')
axes[1].set_ylabel('CDF')
axes[1].set_title('Dystrybuanta skumulowana ocen (CDF)')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/rating_distribution.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 4. Aktywność użytkowników i cold start

In [ ]:
user_counts = df.groupby('UserID')['Rating'].count()
wine_counts = df.groupby('WineID')['Rating'].count()

# Cold start stats
thresholds_u = [3, 5, 10, 20]
thresholds_w = [5, 10, 20, 50]

print('--- Cold Start: użytkownicy ---')
for t in thresholds_u:
    n = (user_counts < t).sum()
    print(f'  < {t:>2} ocen: {n:>5,}  ({n/n_users*100:.1f}%)')

print('\n--- Cold Start: wina ---')
for t in thresholds_w:
    n = (wine_counts < t).sum()
    print(f'  < {t:>2} ocen: {n:>4,}  ({n/n_wines*100:.1f}%)')

print(f'\nŚrednia ocen / user:  {user_counts.mean():.1f}')
print(f'Mediana ocen / user:  {user_counts.median():.0f}')
print(f'Max ocen / user:      {user_counts.max()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram user activity (log scale)
axes[0].hist(user_counts, bins=60, color='steelblue', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_xlabel('Liczba ocen użytkownika')
axes[0].set_ylabel('Liczba użytkowników (skala log)')
axes[0].set_title('Rozkład aktywności użytkowników')
axes[0].axvline(user_counts.median(), color='crimson', linestyle='--',
                label=f'Mediana = {user_counts.median():.0f}')
axes[0].legend()

# Histogram wine activity
axes[1].hist(wine_counts, bins=50, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Liczba ocen wina')
axes[1].set_ylabel('Liczba win')
axes[1].set_title('Rozkład aktywności win')
axes[1].axvline(wine_counts.median(), color='crimson', linestyle='--',
                label=f'Mediana = {wine_counts.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/activity_distributions.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 5. Analiza Long Tail

In [ ]:
wine_sorted  = wine_counts.sort_values(ascending=False)
cumsum_pct   = wine_sorted.cumsum() / wine_sorted.sum()
wines_for_80 = (cumsum_pct <= 0.8).sum()
top20_pct    = cumsum_pct.iloc[int(len(cumsum_pct) * 0.2) - 1]

print(f'Top 20% win pokrywa:       {top20_pct*100:.1f}% ocen')
print(f'80% ocen skupia się na:    {wines_for_80} winach ({wines_for_80/n_wines*100:.1f}% zbioru)')

plt.figure(figsize=(10, 5))
plt.plot(np.arange(len(wine_sorted)), cumsum_pct.values,
         color='steelblue', linewidth=2)
plt.axhline(y=0.8, color='crimson', linestyle='--', label='80% ocen')
plt.axvline(x=wines_for_80, color='crimson', linestyle=':', alpha=0.8)
plt.fill_between(np.arange(wines_for_80), cumsum_pct.values[:wines_for_80],
                 alpha=0.08, color='crimson')
plt.annotate(f'{wines_for_80} win\n({wines_for_80/n_wines*100:.0f}%)',
             xy=(wines_for_80, 0.8), xytext=(wines_for_80+40, 0.65),
             arrowprops=dict(arrowstyle='->', color='crimson'),
             color='crimson', fontsize=10)
plt.xlabel('Wina (od najpopularniejszych)')
plt.ylabel('Skumulowany udział ocen')
plt.title('Analiza Long Tail — koncentracja ocen')
plt.legend()
plt.tight_layout()
plt.savefig('figures/long_tail.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 6. Korelacje

In [ ]:
wine_stats = df.groupby('WineID').agg(
    n_ratings=('Rating', 'count'),
    avg_rating=('Rating', 'mean')
).reset_index()

user_stats = df.groupby('UserID').agg(
    n_ratings=('Rating', 'count'),
    avg_rating=('Rating', 'mean')
).reset_index()

pearson_wine,  p_wine  = stats.pearsonr(wine_stats['n_ratings'], wine_stats['avg_rating'])
spearman_wine, ps_wine = stats.spearmanr(wine_stats['n_ratings'], wine_stats['avg_rating'])
pearson_user,  p_user  = stats.pearsonr(user_stats['n_ratings'], user_stats['avg_rating'])
spearman_user, ps_user = stats.spearmanr(user_stats['n_ratings'], user_stats['avg_rating'])

print('--- Popularność wina vs średnia ocena ---')
print(f'  Pearson  r = {pearson_wine:.4f}  (p = {p_wine:.4e})')
print(f'  Spearman ρ = {spearman_wine:.4f}  (p = {ps_wine:.4e})')
print('\n--- Aktywność usera vs średnia ocena ---')
print(f'  Pearson  r = {pearson_user:.4f}  (p = {p_user:.4e})')
print(f'  Spearman ρ = {spearman_user:.4f}  (p = {ps_user:.4e})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(wine_stats['n_ratings'], wine_stats['avg_rating'],
                alpha=0.35, s=18, color='steelblue')
axes[0].set_xlabel('Liczba ocen (wino)')
axes[0].set_ylabel('Średnia ocena')
axes[0].set_title(f'Popularność wina vs średnia ocena\nPearson r = {pearson_wine:.3f},  Spearman ρ = {spearman_wine:.3f}')

axes[1].scatter(user_stats['n_ratings'], user_stats['avg_rating'],
                alpha=0.15, s=10, color='darkorange')
axes[1].set_xlabel('Liczba ocen (użytkownik)')
axes[1].set_ylabel('Średnia ocena')
axes[1].set_title(f'Aktywność usera vs średnia ocena\nPearson r = {pearson_user:.3f},  Spearman ρ = {spearman_user:.3f}')

plt.tight_layout()
plt.savefig('figures/korelacje_scatter.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 7. Analiza czasowa

In [ ]:
df['Month']     = df['Date'].dt.month
df['Year']      = df['Date'].dt.year
df['DayOfWeek'] = df['Date'].dt.day_name()

monthly_vol = df.groupby(df['Date'].dt.to_period('M')).size()
monthly_avg = df.groupby(df['Date'].dt.to_period('M'))['Rating'].mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

axes[0].plot(monthly_vol.index.to_timestamp(), monthly_vol.values,
             color='steelblue', linewidth=1.2)
axes[0].fill_between(monthly_vol.index.to_timestamp(), monthly_vol.values,
                     alpha=0.15, color='steelblue')
axes[0].set_title('Liczba ocen w czasie (miesięcznie)')
axes[0].set_ylabel('Liczba ocen')
axes[0].tick_params(axis='x', rotation=30)

axes[1].plot(monthly_avg.index.to_timestamp(), monthly_avg.values,
             color='darkorange', linewidth=1.5)
axes[1].axhline(df['Rating'].mean(), color='gray', linestyle='--',
                label=f'Średnia ogólna = {df["Rating"].mean():.2f}')
axes[1].set_title('Trend średniej oceny w czasie')
axes[1].set_ylabel('Średnia ocena')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('figures/ratings_over_time.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
month_labels = ['Sty','Lut','Mar','Kwi','Maj','Cze','Lip','Sie','Wrz','Paź','Lis','Gru']
day_order    = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_labels   = ['Pon','Wt','Śr','Czw','Pt','Sob','Nd']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

month_avg = df.groupby('Month')['Rating'].mean()
axes[0].plot(month_avg.index, month_avg.values, marker='o', color='steelblue', linewidth=2)
axes[0].axhline(df['Rating'].mean(), color='gray', linestyle='--', label='Średnia ogólna')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_labels)
axes[0].set_title('Sezonowość — średnia ocena wg miesiąca')
axes[0].set_ylabel('Średnia ocena')
axes[0].set_ylim(month_avg.min() - 0.05, month_avg.max() + 0.05)
axes[0].legend()

day_avg = df.groupby('DayOfWeek')['Rating'].mean().reindex(day_order)
axes[1].bar(day_labels, day_avg.values, color='steelblue', edgecolor='white')
axes[1].axhline(df['Rating'].mean(), color='gray', linestyle='--', label='Średnia ogólna')
axes[1].set_title('Średnia ocena wg dnia tygodnia')
axes[1].set_ylabel('Średnia ocena')
axes[1].set_ylim(day_avg.min() - 0.05, day_avg.max() + 0.05)
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/seasonality.pdf', bbox_inches='tight', dpi=150)
plt.show()

---
# CZĘŚĆ II — EDA CONTENT FEATURES (wines_df)

## 8. Podgląd i jakość danych win

In [ ]:
print('Shape:', wines_df.shape)
print('\nKolumny:', list(wines_df.columns))
print('\nBraki danych:')
print(wines_df.isnull().sum())
print('\nDuplikaty:', wines_df.duplicated().sum())

In [ ]:
wines_df.head(3)

## 9. Typ wina (Type)

In [ ]:
type_counts = wines_df['Type'].value_counts()
type_pct    = (type_counts / len(wines_df) * 100).round(1)

print('Rozkład typów win:')
for t, n, p in zip(type_counts.index, type_counts.values, type_pct.values):
    print(f'  {t:<20} {n:>4}  ({p}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = sns.color_palette('Set2', len(type_counts))
axes[0].bar(type_counts.index, type_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Liczba win wg typu')
axes[0].set_ylabel('Liczba win')
axes[0].tick_params(axis='x', rotation=25)

axes[1].pie(type_counts.values, labels=type_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title('Udział procentowy typów win')

plt.tight_layout()
plt.savefig('figures/wine_types.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 10. Typ wina a średnia ocena (połączenie z ratings)

In [ ]:
merged = df.merge(wines_df[['WineID','Type','Body','Acidity','ABV','Country']], on='WineID', how='left')

type_rating = merged.groupby('Type')['Rating'].agg(['mean','count','std']).round(3)
type_rating.columns = ['Średnia ocena', 'Liczba ocen', 'Std']
type_rating = type_rating.sort_values('Średnia ocena', ascending=False)
print(type_rating)

plt.figure(figsize=(10, 5))
order = type_rating.index.tolist()
sns.boxplot(data=merged, x='Type', y='Rating', order=order,
            palette='Set2', width=0.5)
plt.axhline(merged['Rating'].mean(), color='crimson', linestyle='--',
            label='Średnia ogólna')
plt.title('Rozkład ocen wg typu wina')
plt.xlabel('Typ wina')
plt.ylabel('Ocena')
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.savefig('figures/type_vs_rating.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 11. Body i Acidity — cechy sensoryczne

In [ ]:
print('Rozkład Body:')
print(wines_df['Body'].value_counts())
print('\nRozkład Acidity:')
print(wines_df['Acidity'].value_counts())

In [ ]:
body_order    = ['Very light-bodied','Light-bodied','Medium-bodied','Medium to full-bodied','Full-bodied','Very full-bodied']
acidity_order = ['Low','Medium','High']

# Filtruj tylko wartości które istnieją w danych
body_order    = [b for b in body_order    if b in wines_df['Body'].values]
acidity_order = [a for a in acidity_order if a in wines_df['Acidity'].values]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

body_counts = wines_df['Body'].value_counts().reindex(body_order).fillna(0)
axes[0].barh(body_order, body_counts.values, color=sns.color_palette('Blues_r', len(body_order)))
axes[0].set_title('Rozkład Body win')
axes[0].set_xlabel('Liczba win')

acid_counts = wines_df['Acidity'].value_counts().reindex(acidity_order).fillna(0)
axes[1].bar(acidity_order, acid_counts.values,
            color=sns.color_palette('Greens_r', len(acidity_order)))
axes[1].set_title('Rozkład Acidity win')
axes[1].set_ylabel('Liczba win')

plt.tight_layout()
plt.savefig('figures/body_acidity.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Body vs średnia ocena
body_rating = merged.groupby('Body')['Rating'].mean().reindex(body_order)
acid_rating = merged.groupby('Acidity')['Rating'].mean().reindex(acidity_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(body_order, body_rating.values,
             color=sns.color_palette('Blues_r', len(body_order)))
axes[0].axvline(merged['Rating'].mean(), color='crimson', linestyle='--')
axes[0].set_title('Średnia ocena wg Body')
axes[0].set_xlabel('Średnia ocena')
axes[0].set_xlim(3.5, 4.2)

axes[1].bar(acidity_order, acid_rating.values,
            color=sns.color_palette('Greens_r', len(acidity_order)))
axes[1].axhline(merged['Rating'].mean(), color='crimson', linestyle='--')
axes[1].set_title('Średnia ocena wg Acidity')
axes[1].set_ylabel('Średnia ocena')
axes[1].set_ylim(3.5, 4.2)

plt.tight_layout()
plt.savefig('figures/body_acidity_vs_rating.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 12. ABV — zawartość alkoholu

In [ ]:
print('ABV — statystyki:')
print(wines_df['ABV'].describe().round(3))

pearson_abv, p_abv = stats.pearsonr(
    merged.dropna(subset=['ABV'])['ABV'],
    merged.dropna(subset=['ABV'])['Rating']
)
print(f'\nKorelacja ABV vs Rating: Pearson r = {pearson_abv:.4f}  (p = {p_abv:.4e})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(wines_df['ABV'].dropna(), bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Rozkład zawartości alkoholu (ABV)')
axes[0].set_xlabel('ABV [%]')
axes[0].set_ylabel('Liczba win')

axes[1].scatter(merged['ABV'], merged['Rating'],
                alpha=0.08, s=5, color='steelblue')
axes[1].set_title(f'ABV vs Ocena  (Pearson r = {pearson_abv:.3f})')
axes[1].set_xlabel('ABV [%]')
axes[1].set_ylabel('Ocena')

plt.tight_layout()
plt.savefig('figures/abv_analysis.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 13. Kraj pochodzenia

In [ ]:
country_counts = wines_df['Country'].value_counts().head(15)
country_rating = merged.groupby('Country')['Rating'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(country_counts.index[::-1], country_counts.values[::-1],
             color='steelblue', edgecolor='white')
axes[0].set_title('Top 15 krajów — liczba win')
axes[0].set_xlabel('Liczba win')

top_countries = country_counts.index
cr = country_rating.reindex(top_countries).sort_values(ascending=True)
axes[1].barh(cr.index, cr.values, color='darkorange', edgecolor='white')
axes[1].axvline(merged['Rating'].mean(), color='crimson', linestyle='--',
                label='Średnia ogólna')
axes[1].set_title('Średnia ocena — top 15 krajów')
axes[1].set_xlabel('Średnia ocena')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/country_analysis.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 14. Odmiany winogron (Grapes)

In [ ]:
def parse_list_column(series):
    """Parsuje kolumnę z listami zapisanymi jako string."""
    result = []
    for val in series.dropna():
        try:
            parsed = ast.literal_eval(val) if isinstance(val, str) else val
            if isinstance(parsed, list):
                result.extend(parsed)
        except Exception:
            pass
    return result

grapes_all  = parse_list_column(wines_df['Grapes'])
grape_counts = Counter(grapes_all)
top_grapes   = pd.Series(dict(grape_counts.most_common(20)))

print(f'Unikalnych odmian winogron: {len(grape_counts)}')
print(f'Top 5: {list(top_grapes.index[:5])}')

plt.figure(figsize=(11, 6))
top_grapes.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 20 odmian winogron w zbiorze')
plt.xlabel('Liczba win')
plt.tight_layout()
plt.savefig('figures/grapes_top20.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Liczba odmian per wino
wines_df['n_grapes'] = wines_df['Grapes'].apply(
    lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0
)
print('Liczba odmian na wino:')
print(wines_df['n_grapes'].value_counts().sort_index())

varietal_pct = (wines_df['n_grapes'] == 1).mean() * 100
print(f'\nWina varietalne (1 odmiana): {varietal_pct:.1f}%')

## 15. Harmonize — pairings z jedzeniem

In [ ]:
harmonize_all   = parse_list_column(wines_df['Harmonize'])
harmonize_counts = Counter(harmonize_all)
top_harmonize    = pd.Series(dict(harmonize_counts.most_common(15)))

print(f'Unikalnych kategorii food pairing: {len(harmonize_counts)}')

plt.figure(figsize=(11, 6))
top_harmonize.sort_values().plot(kind='barh', color='darkorange', edgecolor='white')
plt.title('Top 15 kategorii food pairing (Harmonize)')
plt.xlabel('Liczba win')
plt.tight_layout()
plt.savefig('figures/harmonize_top15.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 16. Heatmapa korelacji — cechy numeryczne

In [ ]:
# Kodowanie porządkowe dla Body i Acidity
body_map = {
    'Very light-bodied': 1, 'Light-bodied': 2, 'Medium-bodied': 3,
    'Medium to full-bodied': 4, 'Full-bodied': 5, 'Very full-bodied': 6
}
acidity_map = {'Low': 1, 'Medium': 2, 'High': 3}

merged['Body_num']    = merged['Body'].map(body_map)
merged['Acidity_num'] = merged['Acidity'].map(acidity_map)

corr_cols = ['Rating', 'ABV', 'Body_num', 'Acidity_num']
corr_matrix = merged[corr_cols].corr(method='pearson').round(3)
corr_matrix.index   = ['Ocena', 'ABV', 'Body', 'Kwasowość']
corr_matrix.columns = ['Ocena', 'ABV', 'Body', 'Kwasowość']

print(corr_matrix)

plt.figure(figsize=(7, 5))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('Macierz korelacji — cechy win vs ocena')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 17. Elaborate — styl produkcji

In [ ]:
# Uproszczenie etykiet Elaborate
wines_df['Elaborate_short'] = wines_df['Elaborate'].apply(
    lambda x: x.split('/')[0] if isinstance(x, str) else 'Unknown'
)
elab_counts = wines_df['Elaborate_short'].value_counts()
print('Styl produkcji (Elaborate):')
print(elab_counts)

merged2 = merged.merge(
    wines_df[['WineID','Elaborate_short']], on='WineID', how='left'
)
elab_rating = merged2.groupby('Elaborate_short')['Rating'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
elab_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Liczba win wg stylu produkcji')
axes[0].set_ylabel('Liczba win')
axes[0].tick_params(axis='x', rotation=20)

elab_rating.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].axhline(merged['Rating'].mean(), color='crimson', linestyle='--')
axes[1].set_title('Średnia ocena wg stylu produkcji')
axes[1].set_ylabel('Średnia ocena')
axes[1].set_ylim(3.5, 4.2)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('figures/elaborate_analysis.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 18. Podsumowanie EDA — tabela zbiorcza content features

In [ ]:
print('=== PODSUMOWANIE — CONTENT FEATURES ===')
print(f'Unikalne typy win:           {wines_df["Type"].nunique()}')
print(f'Unikalne wartości Body:      {wines_df["Body"].nunique()}')
print(f'Unikalne wartości Acidity:   {wines_df["Acidity"].nunique()}')
print(f'Zakres ABV:                  {wines_df["ABV"].min():.1f}% – {wines_df["ABV"].max():.1f}%')
print(f'Unikalne kraje:              {wines_df["Country"].nunique()}')
print(f'Unikalne regiony:            {wines_df["RegionName"].nunique()}')
print(f'Unikalne winiarnie:          {wines_df["WineryName"].nunique()}')
print(f'Unikalne odmiany winogron:   {len(grape_counts)}')
print(f'Unikalne food pairings:      {len(harmonize_counts)}')
print(f'Wina varietalne (1 odmiana): {varietal_pct:.1f}%')
print(f'\nKorelacja ABV vs Ocena:      r = {pearson_abv:.4f}')
print(f'Korelacja Body vs Ocena:     r = {corr_matrix.loc["Body","Ocena"]:.4f}')
print(f'Korelacja Kwasowość vs Ocena:r = {corr_matrix.loc["Kwasowość","Ocena"]:.4f}')